# Microsoft Agent Framework (MAF) — Complete Agentic AI & Multi-Agent Masterclass

**LangGraph → Microsoft Agent Framework feature parity + native Microsoft extras**  
**Research date:** 10 October 2026  
**Language:** Python 3.11+ recommended · **SDK:** `agent-framework-core` + `agent-framework-orchestrations` (+ optional provider packages)

> **Naming:** This is **Microsoft Agent Framework**, NOT the old AutoGen API or Microsoft 365 Agents SDK. Modern examples import `Agent` from `agent_framework` (not old `ChatAgent`).

**Learning strategy**
1. **Offline labs**: Python/Pydantic/SQLite/mock incident simulation; safe without any cloud key.
2. **Native MAF labs**: `Agent`, `@tool`, `AgentSession`, graph executors, `@workflow`, `@step`, workflows and checkpoints.
3. **Live LLM labs**: opt-in, require credentials and may incur costs.
4. **Advanced/optional labs**: orchestration builders, approval, MCP, A2A, Agent Harness, observability and deployment.

Official starting points: [Repository](https://github.com/microsoft/agent-framework) · [Documentation](https://learn.microsoft.com/en-us/agent-framework/) · [Python core](https://github.com/microsoft/agent-framework/tree/main/python/packages/core) · [Official Python samples](https://github.com/microsoft/agent-framework/tree/main/python/samples).

**Safety:** all remediation demos are *mock-only*. No real production service is restarted or modified. Never persist production credentials or untrusted serialized objects in notebook/database outputs.

## 0. Coverage mapping: Your LangGraph classes 41–50 → Microsoft Agent Framework

| Your LangGraph class | Concept | MAF counterpart |
|---|---|---|
| 41 | Intro, graph, node, edges | `WorkflowBuilder`, `Executor`, `@executor`, `@handler`, `WorkflowContext` |
| 42 | Tools and ReAct tool loop | `@tool` + `Agent(tools=[...])`; `agent.run()` |
| 43 | Pydantic, memory | Pydantic `response_format`; `AgentSession` |
| 44 | Memory management | Sessions, context providers, history compaction (provider dependent) |
| 45 | Checkpoints, persistence, LangMem-style long memory | Native workflow checkpoint storage + custom durable memory backend |
| 46 | Human-in-the-loop | `RunContext.request_info` or approval-required tools + resume |
| 47–48 | Subgraphs, orchestration | Nested `WorkflowExecutor`, functional workflows, sequential/concurrent orchestration |
| 49–50 | Multi-agent complete flow / Command | `SequentialBuilder`, `ConcurrentBuilder`, `HandoffBuilder`, `GroupChatBuilder`, `MagenticBuilder`; conditional graph edges |

**Do not claim 1:1 equivalence:** LangGraph `Command`, `Send`, reducers, LangMem, thread/time-travel APIs are separate abstractions. MAF has its own graph, shared state, checkpoints and orchestration primitives. This course demonstrates behavioral counterparts without pretending API equivalence.

## 1. Setup: Python environment, installs, secrets and notebook switches

Recommended terminal setup:

```bash
pip install uv          # if uv is not installed
uv python install 3.11
uv venv .venv --python 3.11
# Windows PowerShell: .venv\Scripts\Activate.ps1
# macOS/Linux: source .venv/bin/activate
uv pip install agent-framework-core agent-framework-orchestrations agent-framework-openai pydantic python-dotenv ipykernel
python -m ipykernel install --user --name maf-env --display-name "Python (MAF)"
```

- For Microsoft Foundry (Azure): `uv pip install agent-framework-foundry azure-identity` then `az login`.
- Alternatively the documented `pip install agent-framework` metapackage installs a standard set.
- Install only official packages relevant to your chosen client; there is **no** required `microsoft-adk` package.
- See [Python install guide](https://github.com/microsoft/agent-framework/blob/main/python/README.md) and [orchestration package](https://github.com/microsoft/agent-framework/tree/main/python/packages/orchestrations).

In [ ]:
# Optional: install into this Jupyter kernel only when you explicitly opt in.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U',
        'agent-framework-core', 'agent-framework-orchestrations', 'agent-framework-openai',
        'pydantic', 'python-dotenv'])
else:
    print('Installation skipped. Use the terminal/venv commands in the cell above.')

In [ ]:
from __future__ import annotations
import os, asyncio, json, sqlite3, time, uuid, math, inspect, importlib.util
from datetime import datetime, timezone
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Annotated, Any, Literal
from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv

load_dotenv()
RUN_LIVE = False      # Explicit opt-in: costs tokens and requires key or Azure login
RUN_ADVANCED = False  # Optional integrations and multi-agent orchestration
RUN_OFFLINE_MAF = True # MAF graph exercises use no model/API calls
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')  # Choose a model enabled in your account
MAF_AVAILABLE = importlib.util.find_spec('agent_framework') is not None
print({'MAF_available': MAF_AVAILABLE, 'RUN_LIVE': RUN_LIVE, 'MODEL': MODEL})

### API keys and clients

**Route A (simplest):** `OPENAI_API_KEY` + `OPENAI_MODEL`; use MAF `OpenAIChatClient` as provider.  
**Route B (enterprise):** Azure CLI `az login`, `FOUNDRY_PROJECT_ENDPOINT` (full project-scoped endpoint) and `FOUNDRY_MODEL` plus `FoundryChatClient`. 

**SDK open-source ≠ model free**. Cloud model and Foundry usage may have charges. We never print keys. Official [provider quickstart](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/README.md).

In [ ]:
# In a notebook, prefer shell environment / .env ignored by git.
# If you must set it interactively, use getpass rather than hardcoding.
if RUN_LIVE and not os.getenv('OPENAI_API_KEY') and not os.getenv('FOUNDRY_PROJECT_ENDPOINT'):
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key (hidden): ')

def readiness():
    return {'framework_installed': MAF_AVAILABLE,
            'openai_key_set': bool(os.getenv('OPENAI_API_KEY')),
            'foundry_project_set': bool(os.getenv('FOUNDRY_PROJECT_ENDPOINT')),
            'azure_cli_option': 'requires az login'}
print(readiness())

## 2. Modern MAF imports, backend selection and helper

**Current API naming:** `Agent(client=..., name=..., instructions=..., tools=...)`, `.run(...)`, `.run(..., stream=True)`, `AgentSession`. `OpenAIChatClient` uses OpenAI Responses model client. Azure `FoundryChatClient` uses a Foundry project. Configure one client; do not accidentally mix API credentials. 

Official [Python README](https://github.com/microsoft/agent-framework/blob/main/python/README.md) and [sample conventions](https://github.com/microsoft/agent-framework/blob/main/python/samples/AGENTS.md).

In [ ]:
if MAF_AVAILABLE:
    import agent_framework
    from agent_framework import Agent, AgentSession, tool
    from agent_framework.openai import OpenAIChatClient
    print('Imported modern MAF Agent, AgentSession, tool, OpenAIChatClient')
else:
    print('MAF not installed. Native MAF cells will be skipped; offline exercises still run.')

def create_client(provider: str = 'openai'):
    if not MAF_AVAILABLE:
        raise RuntimeError('Install agent-framework-core and agent-framework-openai')
    if provider == 'openai':
        return OpenAIChatClient()
    if provider == 'foundry':
        from agent_framework.foundry import FoundryChatClient
        from azure.identity import AzureCliCredential
        return FoundryChatClient(
            project_endpoint=os.environ['FOUNDRY_PROJECT_ENDPOINT'],
            model=os.environ['FOUNDRY_MODEL'],
            credential=AzureCliCredential(),
        )
    raise ValueError('provider must be openai or foundry')

def make_agent(name: str, instructions: str, tools=None, provider='openai'):
    return Agent(client=create_client(provider), name=name, instructions=instructions,
                 tools=list(tools or []))

## 3. First Agent: Hello world

Create and invoke an agent asynchronously. Unlike a LangGraph compiled state graph, this direct call runs an agent without an explicit node graph.

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    hello_agent = make_agent('HelloAgent', 'Explain AI engineering in 3 concise sentences.')
    hello_response = await hello_agent.run('What is an AI agent?')
    print(hello_response.text)
else:
    print('To run live: set RUN_LIVE=True, install MAF and configure credentials.')

## 4. ReAct-style reasoning and tool calling

Agent Framework lets the model select tools and continue the tool-execution loop. **Do not expect the model to reveal private reasoning traces.** A tool can be gated through `approval_mode`; production actions should require approval/policy enforcement. [Tool examples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools).

In [ ]:
if MAF_AVAILABLE:
    @tool(approval_mode='never_require')
    def get_incident_status(incident_id: Annotated[str, 'Incident identifier']) -> str:
        """Return mock status of an IT incident."""
        return json.dumps({'incident_id': incident_id, 'status': 'investigating', 'severity': 'SEV2'})

    @tool(approval_mode='never_require')
    def list_recent_deployments(service: Annotated[str, 'Application service']) -> str:
        """Return fake deployment metadata for lab use."""
        return json.dumps({'service': service, 'latest': '2026-10-10T09:15Z', 'version': 'v2.8.1'})
    print('Tools registered; no cloud calls yet.')
else:
    print('Install MAF to enable @tool examples.')

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    tool_agent = make_agent(
        'ITDiagnosisAgent',
        'You are a read-only SRE assistant. Use tools to obtain facts. Never invent metrics.',
        tools=[get_incident_status, list_recent_deployments]
    )
    print((await tool_agent.run('Check INC-1001 and the checkout-service deployment.')).text)

## 5. Tool schema, type hints, and Pydantic arguments

`@tool` supports descriptions, parameter annotations, and explicit schemas. This matters for predictable tool selection and input validation.

In [ ]:
class SearchIncidentArgs(BaseModel):
    incident_id: str = Field(pattern=r'^INC-\d{4}$')
    include_logs: bool = False

valid_args = SearchIncidentArgs.model_validate({'incident_id': 'INC-1001', 'include_logs': True})
print(valid_args.model_dump())
try:
    SearchIncidentArgs.model_validate({'incident_id': 'bad'})
except ValidationError as error:
    print('Validation catches invalid incident IDs:', error.errors()[0]['type'])

In [ ]:
if MAF_AVAILABLE:
    @tool(name='search_incident', description='Fetch a mock incident by ID',
          schema=SearchIncidentArgs, approval_mode='never_require')
    def search_incident(incident_id: str, include_logs: bool = False) -> str:
        """Search the training-only incident catalog."""
        return json.dumps({'id': incident_id, 'include_logs': include_logs, 'found': True})
    print('Schema-backed @tool created.')

## 6. Structured output with Pydantic: `response_format` (not LangChain parser)

Current MAF Python accepts `options={'response_format': MyModel}` on `agent.run`. When supported by the model/client, `result.value` is the parsed Pydantic model. [Official Microsoft structured-output guide](https://learn.microsoft.com/en-us/agent-framework/agents/structured-outputs).

In [ ]:
class IncidentTriage(BaseModel):
    incident_id: str
    severity: Literal['SEV1', 'SEV2', 'SEV3', 'SEV4']
    summary: str
    affected_services: list[str]
    needs_human_approval: bool

mock_triage = IncidentTriage(incident_id='INC-1001', severity='SEV2',
    summary='Checkout 5xx spike after deployment', affected_services=['checkout'],
    needs_human_approval=True)
print(mock_triage.model_dump_json(indent=2))

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    triage_agent = make_agent('TriageAgent', 'Extract incident triage. Return factual structured fields.')
    triage_result = await triage_agent.run(
        'INC-1001: checkout reports elevated 5xx immediately after version 2.8.1 rollout.',
        options={'response_format': IncidentTriage},
    )
    print(triage_result.value.model_dump() if triage_result.value else triage_result.text)

## 7. Multi-turn conversational memory using `AgentSession`

A direct agent is stateless unless you supply a session. `session = agent.create_session()` and pass `session=session` on every turn. Sessions are not identical to graph workflow checkpoints. [Official sessions guide](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/).

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    chat_agent = make_agent('ConversationAgent', 'Remember only the supplied session conversation.')
    session = chat_agent.create_session()
    print((await chat_agent.run('Our service is called checkout-service.', session=session)).text)
    print((await chat_agent.run('What is the service called?', session=session)).text)

## 8. Serialize/restore sessions (conversation persistence)

`AgentSession.to_dict()` and `AgentSession.from_dict(...)` serialize session state; suitable backing stores include secure SQL/NoSQL/Redis systems with appropriate isolation and retention. **Do not confuse a saved conversation with replayable workflow state.**

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    session_state = session.to_dict()
    print('Serialized session keys:', sorted(session_state.keys()))
    # Save only to a protected, tenant-scoped store in real deployments.
    restored_session = AgentSession.from_dict(session_state)
    print((await chat_agent.run('Repeat the service name.', session=restored_session)).text)

## 9. Offline conversation memory and truncation policies

This offline implementation teaches retention strategies without claiming it is MAF's built-in compaction API. Discuss token limits, safety of summaries, tenant isolation, deletion and PII redaction.

In [ ]:
def retain_recent_turns(turns: list[dict], max_turns: int = 4) -> list[dict]:
    """A learning policy, not MAF internal chat-memory implementation."""
    if max_turns <= 0:
        return []
    return turns[-max_turns:]

sample_turns = [{'speaker': 'user' if i % 2 == 0 else 'assistant', 'text': f'Turn {i}'} for i in range(10)]
assert len(retain_recent_turns(sample_turns, 4)) == 4
print(retain_recent_turns(sample_turns))

## 10. Custom long-term memory: SQLite (LangMem-like *pattern*)

MAF supports context and memory integrations; this notebook uses a **custom application-owned SQLite memory** for deterministic teaching. It is **not** `LangMem` and not MAF's native `ContextProvider` API. Connect this store to agents via appropriately scoped tools or a custom context provider.

In [ ]:
MEMORY_DB = Path('maf_long_term_memory_demo.sqlite3')
with sqlite3.connect(MEMORY_DB) as con:
    con.execute('''CREATE TABLE IF NOT EXISTS memory (
        namespace TEXT NOT NULL, memory_key TEXT NOT NULL, memory_value TEXT NOT NULL,
        updated_utc TEXT NOT NULL, PRIMARY KEY (namespace, memory_key))''')

def remember(namespace: str, key: str, value: str):
    """Namespace MUST be derived from verified tenant identity in production."""
    with sqlite3.connect(MEMORY_DB) as con:
        con.execute('INSERT INTO memory VALUES (?,?,?,?) '
                    'ON CONFLICT(namespace,memory_key) DO UPDATE SET memory_value=excluded.memory_value, updated_utc=excluded.updated_utc',
                    (namespace, key, value, datetime.now(timezone.utc).isoformat()))

def recall(namespace: str, key: str) -> str | None:
    with sqlite3.connect(MEMORY_DB) as con:
        row = con.execute('SELECT memory_value FROM memory WHERE namespace=? AND memory_key=?', (namespace, key)).fetchone()
    return row[0] if row else None

remember('tenant-demo-01', 'service_owner', 'platform-team')
assert recall('tenant-demo-01', 'service_owner') == 'platform-team'
assert recall('another-tenant', 'service_owner') is None
print('Saved mock memory; cross-tenant namespace lookup returns None')

In [ ]:
if MAF_AVAILABLE:
    @tool(approval_mode='never_require')
    def read_service_owner(service_name: Annotated[str, 'Name of service']) -> str:
        """Read service owner from a pre-authorized demo memory scope."""
        # Demo scope is intentionally fixed; real auth must supply tenant from signed identity.
        return recall('tenant-demo-01', 'service_owner') or 'unknown'
    print('Read-only memory tool available for MAF agents.')

## 11. State, context, and middleware vs LangGraph reducers

The LangGraph state dictionary/reducer model is not the same as MAF's workflow messages, executor-scoped state and agent context providers. You can also inject context via middleware; build strict data contracts rather than assuming mutable shared global state.

Sources: [State management samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/state-management) · [Middleware](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/middleware) · [Context providers](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/context_providers).

In [ ]:
@dataclass
class IncidentState:
    incident_id: str
    summary: str
    severity: str = 'SEV3'
    logs: str = ''
    metrics: str = ''
    deployment: str = ''
    action_plan: str = ''
    approved: bool = False

state = IncidentState(incident_id='INC-1001', summary='Checkout 5xx rate rising')
assert state.approved is False
print(asdict(state))

## 12. Native MAF graph: Executors, edges, WorkflowBuilder

Native graph workflow is a particularly close counterpart to LangGraph classes 41/47/48. A node receives typed input and `WorkflowContext`; it uses `await ctx.send_message()` to send downstream values or `await ctx.yield_output()` for workflow outputs. This example needs the MAF package but **no model or API key**.

Source: [Official step1_executors_and_edges.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/_start-here/step1_executors_and_edges.py).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    from agent_framework import WorkflowBuilder, WorkflowContext, executor
    from typing_extensions import Never

    @executor(id='normalize_incident')
    async def normalize_incident(text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text.strip().upper())

    @executor(id='publish_incident')
    async def publish_incident(text: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f'NORMALIZED: {text}')

    basic_graph = (WorkflowBuilder(start_executor=normalize_incident)
                   .add_edge(normalize_incident, publish_incident).build())
    graph_result = await basic_graph.run('  inc-1001  ')
    print(graph_result.get_outputs())
else:
    print('Offline MAF graph skipped; install agent-framework-core and set RUN_OFFLINE_MAF=True.')

## 13. Custom Executor class and `@handler`

Use class executors when steps need instance-scoped configuration or lifecycle hooks; `@executor` function steps work for straightforward transformations. The *handler signature and output type* are part of the graph contract.

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    from agent_framework import Executor, handler

    class ClassifySeverity(Executor):
        def __init__(self, id='classify_severity'):
            super().__init__(id=id)

        @handler
        async def classify(self, text: str, ctx: WorkflowContext[str]) -> None:
            sev = 'SEV1' if 'OUTAGE' in text.upper() else 'SEV3'
            await ctx.send_message(f'{sev}: {text}')

    # Construct a single instance per topology (don't create two different objects for one node).
    node = ClassifySeverity()
    classifier_graph = WorkflowBuilder(start_executor=node).add_edge(node, publish_incident).build()
    print((await classifier_graph.run('Checkout outage')).get_outputs())

## 14. Conditional routing: graph edges vs LangGraph conditional edges

`WorkflowBuilder.add_edge(source, target, condition=predicate)` allows deterministic routing. Conditions evaluate a message before dispatch. We use a typed string in this offline exercise.

Source: [Official edge_condition.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/control-flow/edge_condition.py).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    @executor(id='severity_router')
    async def severity_router(msg: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(msg)

    @executor(id='sev1_lane')
    async def sev1_lane(msg: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output('PAGE ONCALL: ' + msg)

    @executor(id='sev3_lane')
    async def sev3_lane(msg: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output('CREATE TICKET: ' + msg)

    conditional_graph = (WorkflowBuilder(start_executor=severity_router)
        .add_edge(severity_router, sev1_lane, condition=lambda m: m.startswith('SEV1'))
        .add_edge(severity_router, sev3_lane, condition=lambda m: not m.startswith('SEV1'))
        .build())
    print((await conditional_graph.run('SEV1 checkout outage')).get_outputs())

## 15. Functional workflow: native Python control flow + `@workflow`

MAF supports code-first deterministic workflows. `@workflow` can wrap ordinary async Python steps (branching, loops, `asyncio.gather`). Avoid inventing a LangGraph-like reducer when a typed Python function is sufficient.

Source: [Functional workflow official examples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/functional).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    from agent_framework import workflow

    @workflow
    async def functional_route(raw: str) -> str:
        normalized = raw.strip().lower()
        if 'outage' in normalized:
            return 'priority=p1;route=incident-commander'
        return 'priority=p3;route=service-desk'

    print((await functional_route.run('Checkout outage')).get_outputs())

## 16. Parallel fan-out / fan-in: native Python `asyncio.gather`

This is the functional equivalent of parallel nodes. For orchestration with MAF agents, see `ConcurrentBuilder` in a later module. Never share mutable objects unsafely across concurrent tasks.

In [ ]:
async def fetch_fake_logs(service: str) -> dict:
    await asyncio.sleep(0.01)
    return {'kind': 'logs', 'service': service, 'signal': 'HTTP 500 rate 12%'}

async def fetch_fake_metrics(service: str) -> dict:
    await asyncio.sleep(0.01)
    return {'kind': 'metrics', 'service': service, 'signal': 'p95 latency 2800ms'}

async def fetch_fake_deploy(service: str) -> dict:
    await asyncio.sleep(0.01)
    return {'kind': 'deploy', 'service': service, 'signal': 'v2.8.1 deployed recently'}

parallel_evidence = await asyncio.gather(
    fetch_fake_logs('checkout'), fetch_fake_metrics('checkout'), fetch_fake_deploy('checkout'))
assert len(parallel_evidence) == 3
print(json.dumps(parallel_evidence, indent=2))

## 17. Bounded evaluator–optimizer loops

Avoid infinite self-critique. Always configure max rounds/time/cost and objective stopping conditions. Here the evaluator is deterministic; swap its implementation for a real evaluator agent if desired.

In [ ]:
def quality_score(draft: str) -> float:
    markers = ('impact', 'cause', 'action', 'verification')
    return sum(k in draft.lower() for k in markers) / len(markers)

plan = 'Impact: checkout failures. Cause: suspect deployment.'
for attempt in range(1, 4):
    if quality_score(plan) == 1.0:
        break
    if 'action' not in plan.lower():
        plan += ' Action: request human approval for canary rollback.'
    if 'verification' not in plan.lower():
        plan += ' Verification: inspect 5xx/error budget after simulated rollback.'
print({'attempts': attempt, 'score': quality_score(plan), 'plan': plan})

## 18. Native checkpointing with `@step` and `InMemoryCheckpointStorage`

`@step` captures step results and supports replay/resume. `InMemoryCheckpointStorage` is for demos; for persistence use supported durable storage and proper tenancy. **Never describe in-memory checkpoints as crash-durable.**

Source: [Official steps_and_checkpointing.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/functional/steps_and_checkpointing.py).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    from agent_framework import InMemoryCheckpointStorage, step
    demo_checkpoint_store = InMemoryCheckpointStorage()
    calls = {'fetch': 0}

    @step
    async def expensive_mock_fetch(ref: str) -> str:
        calls['fetch'] += 1
        return f'raw evidence for {ref}'

    @workflow(checkpoint_storage=demo_checkpoint_store)
    async def checkpointed_pipeline(ref: str) -> str:
        evidence = await expensive_mock_fetch(ref)
        return evidence.upper()

    first_checkpoint_run = await checkpointed_pipeline.run('INC-1001')
    checkpoint_record = await demo_checkpoint_store.get_latest(workflow_name='checkpointed_pipeline')
    print('Initial:', first_checkpoint_run.get_outputs(), '| fetch calls:', calls['fetch'])
    if checkpoint_record is not None:
        replay = await checkpointed_pipeline.run(checkpoint_id=checkpoint_record.checkpoint_id)
        print('Restored:', replay.get_outputs(), '| fetch calls:', calls['fetch'])

## 19. Human-in-the-loop (HITL): `RunContext.request_info()`

Native MAF functional workflow can suspend and resume on human input. `@step` prevents completed expensive steps rerunning on resume. Real apps should **persist checkpoints** for requests that may survive process restarts.

Source: [Official functional HITL example](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/functional/hitl_review.py).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    from agent_framework import RunContext, WorkflowRunState

    @step
    async def create_mock_remediation_proposal(incident: str) -> str:
        return f'SIMULATED rollback candidate for {incident}; NO changes executed'

    @workflow
    async def approval_workflow(incident: str, ctx: RunContext) -> str:
        proposal = await create_mock_remediation_proposal(incident)
        decision = await ctx.request_info(
            {'proposal': proposal, 'instructions': 'Type APPROVE or REJECT'},
            response_type=str, request_id='approval-incident')
        return ('Approved for SIMULATION only: ' if decision == 'APPROVE' else 'Rejected: ') + proposal

    pending = await approval_workflow.run('INC-1001')
    print('Paused state:', pending.get_final_state())
    print('Request IDs:', [e.request_id for e in pending.get_request_info_events()])
    # Instructor chooses a deterministic decision here; no interactive prompt needed.
    approved_result = await approval_workflow.run(responses={'approval-incident': 'APPROVE'})
    print('Output:', approved_result.get_outputs())

## 20. Durable file/checkpoint storage, Cosmos DB, and resume considerations

Native MAF supports workflow checkpoint stores beyond in-memory, including file and Cosmos DB sample integrations. Persistent checkpoint **schemas, locations, retention, encryption, tenant keys, and execution idempotency** matter. Resuming a workflow is distinct from simply re-asking an LLM to continue.

Read [MAF checkpoints](https://learn.microsoft.com/en-us/agent-framework/workflows/checkpoints) · [Checkpoint demos](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/checkpoint).

In [ ]:
if MAF_AVAILABLE:
    from agent_framework import FileCheckpointStorage
    print('FileCheckpointStorage imported:', FileCheckpointStorage.__name__)
    print('Read exact current constructor signature before configuring a protected checkpoint directory:')
    print(inspect.signature(FileCheckpointStorage))

## 21. Subgraphs / nested workflows: `WorkflowExecutor`

Microsoft Agent Framework supports child-workflow composition, like LangGraph subgraphs, using `WorkflowExecutor` and typed parent/child result mapping.

Source: [Official sub_workflow_basics.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/composition/sub_workflow_basics.py). Before production, use the sample's exact request/output forwarding semantics—do not assume child output automatically propagates to parent.

In [ ]:
if MAF_AVAILABLE:
    from agent_framework import WorkflowExecutor
    print('Sub-workflow integration object:', WorkflowExecutor)
    print('Constructor signature:', inspect.signature(WorkflowExecutor))
else:
    print('Sub-workflow integration requires MAF installed.')

## 22. Multi-agent orchestration 1: `SequentialBuilder`

This high-level orchestrator passes collaboration context from agent to agent. Use it for *plan → analyze → review* pipelines. Requires `agent-framework-orchestrations` and configured LLM credentials.

Source: [Official sequential_agents.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/sequential_agents.py).

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    from agent_framework.orchestrations import SequentialBuilder
    planner = make_agent('Planner', 'Create a short incident diagnostic plan without actions.')
    reviewer = make_agent('Reviewer', 'Review the previous plan for gaps and safety; be concise.')
    sequence = SequentialBuilder(participants=[planner, reviewer]).build()
    sequence_result = await sequence.run('Checkout 5xx after latest deploy; suggest investigation.')
    for output in sequence_result.get_outputs():
        print(output)

## 23. Multi-agent orchestration 2: `ConcurrentBuilder`

Multiple specialists assess the same incident independently; their findings are aggregated. This is native agent fan-out / fan-in. Prefer read-only tools for parallel investigation.

Source: [Official concurrent_agents.py](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/concurrent_agents.py).

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    from agent_framework.orchestrations import ConcurrentBuilder
    log_specialist = make_agent('LogSpecialist', 'Discuss likely logging evidence, noting missing data.')
    metrics_specialist = make_agent('MetricsSpecialist', 'Discuss metrics to inspect, no invented values.')
    deploy_specialist = make_agent('DeploySpecialist', 'Discuss release correlation, no unauthorized changes.')
    specialists = [log_specialist, metrics_specialist, deploy_specialist]
    fanout = ConcurrentBuilder(participants=specialists).build()
    fanout_result = await fanout.run('INC-1001: checkout 5xx increased after deploy.')
    print([str(x) for x in fanout_result.get_outputs()])

## 24. Multi-agent orchestration 3: `HandoffBuilder`

**Handoff** transfers control to a specialist (different from a supervisor that merely calls agents as tools). Handoff orchestration has policies for termination/turn-taking; versioned builder signatures may evolve. Read the official sample and inspect the live SDK before customizing.

Source: [Official handoff examples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/orchestrations) and [orchestration package README](https://github.com/microsoft/agent-framework/blob/main/python/packages/orchestrations/README.md).

In [ ]:
if MAF_AVAILABLE:
    from agent_framework.orchestrations import HandoffBuilder
    print('HandoffBuilder:', inspect.signature(HandoffBuilder))
    print('Builder methods:', [n for n in dir(HandoffBuilder) if n.startswith(('add_', 'with_', 'build'))])

## 25. Multi-agent orchestration 4: `GroupChatBuilder`

Group chat uses manager/speaker selection and shared conversation context. Guard against endless discussions with max turn limits and quality checks. The manager may require a specific structured selection response contract.

Source: [MAF orchestration samples](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/README.md).

In [ ]:
if MAF_AVAILABLE:
    from agent_framework.orchestrations import GroupChatBuilder
    print('GroupChatBuilder API:', inspect.signature(GroupChatBuilder))
    print('Methods:', [n for n in dir(GroupChatBuilder) if n in ('set_manager','participants','build') or n.startswith('with_')])

if RUN_LIVE and RUN_ADVANCED and MAF_AVAILABLE:
    group_manager = make_agent('GroupManager',
        'Coordinate short conversation. Ask each specialist to contribute and finish within 4 turns.')
    group_researcher = make_agent('GroupResearcher', 'Extract evidence, distinguish known from unknown.')
    group_writer = make_agent('GroupWriter', 'Write a concise incident summary from the discussion.')
    group_chat_demo = GroupChatBuilder(
        participants=[group_researcher, group_writer],
        orchestrator_agent=group_manager,
        termination_condition=lambda messages: sum(1 for m in messages if m.role == 'assistant') >= 4,
        intermediate_output_from=[group_researcher, group_writer],
    ).build()
    group_chat_result = await group_chat_demo.run('Summarize safe incident-response best practices.')
    print(group_chat_result.get_outputs())

## 26. Multi-agent orchestration 5: `MagenticBuilder`

Magentic is a task-oriented orchestrator designed for complex planning/collaboration. This is not a synonym for every LangGraph supervisor pattern. Treat it as an advanced lab because it can consume multiple model turns.

Source: [MAF orchestrations](https://github.com/microsoft/agent-framework/blob/main/python/packages/orchestrations/README.md).

In [ ]:
if MAF_AVAILABLE:
    from agent_framework.orchestrations import MagenticBuilder
    print('MagenticBuilder API:', inspect.signature(MagenticBuilder))
    print('Builder methods:', [n for n in dir(MagenticBuilder) if n.startswith(('with_', 'build'))])

if RUN_LIVE and RUN_ADVANCED and MAF_AVAILABLE:
    magentic_manager = make_agent('MagenticManager', 'Coordinate a focused read-only incident report.')
    magentic_researcher = make_agent('MagenticResearcher', 'Gather evidence from provided data only.')
    magentic_reviewer = make_agent('MagenticReviewer', 'Check conclusions for overclaiming.')
    magentic_demo = MagenticBuilder(
        participants=[magentic_researcher, magentic_reviewer],
        manager_agent=magentic_manager, max_round_count=4,
        max_stall_count=2, max_reset_count=1,
    ).build()
    magentic_result = await magentic_demo.run(
        'Summarize root cause hypotheses and safe next checks for a checkout 5xx spike.')
    print(magentic_result.get_outputs())

## 27. Orchestration comparison / how to choose

| Pattern | Best fit | Typical tradeoff |
|---|---|---|
| Graph `WorkflowBuilder` | Typed, predictable routing | More wiring |
| Functional `@workflow` | Python-native deterministic control | You own branching logic |
| Sequential | Draft → verify → publish | Added latency |
| Concurrent | Independent specialist analysis | Higher simultaneous token cost |
| Handoff | Dynamic specialist ownership | Harder to audit control transfers |
| Group chat | Negotiation / brainstorming | Turn explosion risk |
| Magentic | Complex task planning / iteration | Highest complexity/cost potential |
| Nested workflow | Reusable subsystems | State/checkpoint boundaries to design |

**Rule:** start with one agent; add multiple agents only if specialization and measured quality justify orchestration overhead.

In [ ]:
def select_pattern(task_type: str, parallelizable: bool, dynamic_owner: bool) -> str:
    if dynamic_owner:
        return 'handoff'
    if parallelizable:
        return 'concurrent'
    if task_type == 'business_process':
        return 'sequential'
    return 'single_agent_or_functional_workflow'

for args in [('incident', True, False), ('support', False, True), ('business_process', False, False)]:
    print(args, '→', select_pattern(*args))

# Part II — Enterprise capstone: Multi-Agent IT Incident Resolution

**Architecture:** `Triage → [Logs || Metrics || Deployments] → Root-cause synthesis → Remediation plan → Human approval → Simulated action → Verification → Audit`.

**Training safety constraints:** no real API/SSH/kubectl/cloud changes; no untrusted arbitrary shell execution; plan-only outputs; human approval is required even to invoke mock action handler. We show both an **offline fully executable state machine** and a **native MAF multi-agent builder extension** for live LLM runs.

## 28. Mock incident corpus and read-only tools

In [ ]:
INCIDENTS = {
    'INC-1001': {
        'service': 'checkout', 'symptom': '5xx spike', 'severity': 'SEV2',
        'logs': ['POST /checkout 500: upstream connection reset', 'timeout calling payment-provider'],
        'metrics': {'error_rate_pct': 12.4, 'p95_ms': 2800, 'baseline_error_rate_pct': 0.3},
        'deploy': {'version': '2.8.1', 'minutes_before_incident': 7},
    }
}

def fetch_incident(incident_id: str) -> dict:
    if incident_id not in INCIDENTS:
        raise KeyError(f'Unknown training incident: {incident_id}')
    return INCIDENTS[incident_id]

def read_logs(incident_id: str) -> list[str]:
    return fetch_incident(incident_id)['logs']

def read_metrics(incident_id: str) -> dict:
    return fetch_incident(incident_id)['metrics']

def read_deployment(incident_id: str) -> dict:
    return fetch_incident(incident_id)['deploy']

assert len(read_logs('INC-1001')) == 2
print(json.dumps(fetch_incident('INC-1001'), indent=2))

## 29. Capstone schema: every stage has a typed contract

In [ ]:
class EvidenceBundle(BaseModel):
    incident_id: str
    logs: list[str]
    metrics: dict[str, float]
    deployment: dict[str, Any]

class RemediationProposal(BaseModel):
    incident_id: str
    suspected_cause: str
    evidence: list[str]
    recommended_action: str
    risk_level: Literal['low','medium','high']
    approval_required: bool = True

class IncidentAudit(BaseModel):
    incident_id: str
    approved: bool
    executed: bool
    verification: str
    notes: str

print('Pydantic contracts:', [c.__name__ for c in [EvidenceBundle, RemediationProposal, IncidentAudit]])

## 30. Capstone: parallel collection, synthesis and a proposal (offline)

As a lab, the deterministic synthesized hypothesis is intentionally labelled **suspected**, not a guaranteed root cause. Human engineer must correlate actual evidence. No network calls.

In [ ]:
async def collect_training_evidence(incident_id: str) -> EvidenceBundle:
    async def get_logs():
        await asyncio.sleep(0.01)
        return read_logs(incident_id)
    async def get_metrics():
        await asyncio.sleep(0.01)
        return read_metrics(incident_id)
    async def get_deploy():
        await asyncio.sleep(0.01)
        return read_deployment(incident_id)
    logs, metrics, deployment = await asyncio.gather(get_logs(), get_metrics(), get_deploy())
    return EvidenceBundle(incident_id=incident_id, logs=logs, metrics=metrics, deployment=deployment)

def draft_training_proposal(evidence: EvidenceBundle) -> RemediationProposal:
    deploy = evidence.deployment
    return RemediationProposal(
        incident_id=evidence.incident_id,
        suspected_cause='Possible regression related to recent deployment',
        evidence=[f"Error rate {evidence.metrics['error_rate_pct']}%", f"Deploy {deploy['version']} occurred {deploy['minutes_before_incident']}m earlier"],
        recommended_action='Simulate a canary rollback review; do not roll back production without authorization',
        risk_level='high', approval_required=True,
    )

evidence = await collect_training_evidence('INC-1001')
proposal = draft_training_proposal(evidence)
print(proposal.model_dump_json(indent=2))

## 31. Capstone: human approval, simulated remediation, verification and audit

`approval=True` below represents a **human-approved lab scenario**, not a real authorization claim. In production approvals must be authenticated, logged, least-privileged, tied to exact action/parameters and have expiry/replay protection.

In [ ]:
MOCK_EXECUTION_LOG: list[dict[str, Any]] = []

def simulate_remediation(proposal: RemediationProposal, approval: bool) -> IncidentAudit:
    if not approval:
        return IncidentAudit(incident_id=proposal.incident_id, approved=False, executed=False,
            verification='not-run', notes='Human rejected proposed operation')
    event = {'incident': proposal.incident_id, 'mode': 'simulation-only',
             'action': proposal.recommended_action, 'at_utc': datetime.now(timezone.utc).isoformat()}
    MOCK_EXECUTION_LOG.append(event)
    return IncidentAudit(incident_id=proposal.incident_id, approved=True, executed=True,
                         verification='mock-health-check-passed',
                         notes='No actual infrastructure was touched')

approved_demo = simulate_remediation(proposal, approval=True)
rejected_demo = simulate_remediation(proposal, approval=False)
assert approved_demo.executed and not rejected_demo.executed
print('APPROVED:', approved_demo.model_dump())
print('REJECTED:', rejected_demo.model_dump())
print('Mock event log:', MOCK_EXECUTION_LOG[-1])

## 32. Capstone MAF multi-agent version (live opt-in)

This expands the offline pipeline into MAF: specialist agents run in parallel with `ConcurrentBuilder`, and a synthesizer reviews their findings. Approval and execution intentionally remain governed by the offline simulation policy. Avoid wiring a model directly to an irreversible production action.

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    from agent_framework.orchestrations import ConcurrentBuilder
    def new_readonly_agent(name, focus):
        return make_agent(name, f'You are a read-only incident specialist. Focus on {focus}. '
                         'Only use given evidence. Mark unknowns. Never perform writes.')
    incident_agents = [
        new_readonly_agent('LogAgent','log evidence'),
        new_readonly_agent('MetricsAgent','service metrics'),
        new_readonly_agent('DeploymentAgent','recent release risk'),
    ]
    multiagent_investigation = ConcurrentBuilder(participants=incident_agents).build()
    sample_incident_text = json.dumps(fetch_incident('INC-1001'))
    investigation_run = await multiagent_investigation.run(sample_incident_text)
    investigation_findings = '\n'.join(str(x) for x in investigation_run.get_outputs())
    synthesizer = make_agent('RootCauseSynthesizer',
                             'Summarize hypotheses and unknowns. Never claim definitive causality.')
    synthesis = await synthesizer.run(investigation_findings)
    print(synthesis.text)

## 33. Optional advanced: native `WorkflowBuilder` end-to-end topology

Build a graph from ingestion executor → triage executor → two conditional lanes → publication executor(s). Add checkpoints, request/response approvals and nested workflows once basic contracts are stable. For genuine parallel fan-out/fan-in, see native `parallelism/fan_out_fan_in_edges.py`; simply adding two outgoing edges is not a safe substitute for an aggregation join.

Official: [MAF parallel graph samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/parallelism) · [full orchestration samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows).

In [ ]:
if MAF_AVAILABLE and RUN_OFFLINE_MAF:
    @executor(id='capstone_entry')
    async def capstone_entry(incident_id: str, ctx: WorkflowContext[str]) -> None:
        record = fetch_incident(incident_id)
        await ctx.send_message(f"{incident_id}|{record['severity']}|{record['service']}")

    @executor(id='capstone_dispatch')
    async def capstone_dispatch(message: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(message)

    @executor(id='capstone_sev2')
    async def capstone_sev2(message: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output('ROUTE: on-call incident commander: ' + message)

    @executor(id='capstone_other')
    async def capstone_other(message: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output('ROUTE: service desk: ' + message)

    maf_capstone_graph = (WorkflowBuilder(start_executor=capstone_entry)
        .add_edge(capstone_entry, capstone_dispatch)
        .add_edge(capstone_dispatch, capstone_sev2, condition=lambda m: '|SEV2|' in m)
        .add_edge(capstone_dispatch, capstone_other, condition=lambda m: '|SEV2|' not in m)
        .build())
    print((await maf_capstone_graph.run('INC-1001')).get_outputs())

# Part III — Capabilities beyond your LangGraph syllabus

Now explore Microsoft-specific enterprise capabilities and advanced features. Some are core SDK features; some require integrations/services and/or preview packages.

## 34. Streaming: `agent.run(..., stream=True)` and events

Stream agent token updates or workflow events. Streaming does not make tool execution itself instant; measure **time-to-first-token** separately from end-to-end latency.

In [ ]:
if RUN_LIVE and MAF_AVAILABLE:
    streaming_agent = make_agent('Streamer','Explain concepts in clear, concise English.')
    async for update in streaming_agent.run('What is agent orchestration?', stream=True):
        if update.text:
            print(update.text, end='', flush=True)
    print()

## 35. Guardrails, tool approval, dynamic tool exposure

Core MAF includes `@tool(approval_mode='always_require')` so sensitive tools can pause for approval. SDK also supports approval middleware and tool caps. **Do not auto-approve production-changing tools**. Tool approval != full prompt-injection defense.

Official samples: [tool approval](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools) · [tool execution limits](https://github.com/microsoft/agent-framework/blob/main/python/samples/02-agents/tools/control_total_tool_executions.py) · [dynamic tool exposure](https://github.com/microsoft/agent-framework/blob/main/python/samples/02-agents/tools/dynamic_tool_exposure.py).

In [ ]:
if MAF_AVAILABLE:
    @tool(approval_mode='always_require')
    def propose_change_ticket(incident_id: Annotated[str, 'Incident to change']) -> str:
        """DEMO: generate a mock change ticket after explicit tool approval."""
        return f'MOCK ticket created for {incident_id}; NO external mutation'
    print('Approval-required tool declared; not invoked automatically.')

In [ ]:
# Deterministic input guardrails complement framework approvals.
def validate_incident_id(value: str) -> str:
    import re
    if not re.fullmatch(r'INC-\d{4}', value):
        raise ValueError('Only incident identifiers in INC-1234 format are accepted')
    return value

assert validate_incident_id('INC-1001') == 'INC-1001'
for candidate in ['INC-1001; rm -rf /', '../INC-1001']:
    try:
        validate_incident_id(candidate)
    except ValueError:
        print('Rejected unsafe input:', candidate)

## 36. Observability, tracing and OpenTelemetry

Production tracing should capture model/tool durations, errors, model selection, request IDs, agent hops, and cost estimates **without logging sensitive user data by default**. Microsoft Agent Framework has OpenTelemetry integration samples; connect to your approved collector, Application Insights, Azure Monitor, or other OTel backend.

Source: [Observability samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/observability).

In [ ]:
AUDIT_EVENTS = []

def audit_event(name: str, **safe_fields):
    """Only pass approved non-sensitive identifiers/metadata."""
    evt = {'event': name, 'ts': datetime.now(timezone.utc).isoformat(), **safe_fields}
    AUDIT_EVENTS.append(evt)
    return evt

audit_event('investigation_started', incident_id='INC-1001', mode='simulated')
audit_event('approval_required', incident_id='INC-1001', mode='simulated')
print(json.dumps(AUDIT_EVENTS, indent=2))

## 37. Evaluation: correctness, faithfulness, tool-use policy, latency and approvals

Always evaluate single- and multi-agent systems against a golden dataset. Add regression cases for hallucinated metrics, unsupported root causes, unsafe tool calls, rejected approvals, and tenant data leakage. The following offline tests are deterministic and do not require paid model calls.

In [ ]:
GOLDEN_CASES = [
    {'input': 'INC-1001', 'expected_route': 'oncall', 'approved': False},
    {'input': 'INC-1001', 'expected_route': 'oncall', 'approved': True},
]

def route_from_incident(incident_id: str) -> str:
    return 'oncall' if fetch_incident(incident_id)['severity'] in ('SEV1','SEV2') else 'service_desk'

passed = 0
for case in GOLDEN_CASES:
    observed = route_from_incident(case['input'])
    approved = simulate_remediation(proposal, approval=case['approved'])
    assert observed == case['expected_route']
    assert approved.executed == case['approved']
    passed += 1
print(f'{passed}/{len(GOLDEN_CASES)} policy and routing tests passed')

## 38. Token limits, max iterations, retries, deadlines and idempotency

Bounded retries prevent retry storms. Tool re-execution during recovery is dangerous unless side effects are idempotent or protected by action IDs. `max_iterations` and tool invocation limits are supported in relevant MAF options; inspect exact model/provider support.

In [ ]:
async def simulated_api_with_retry(max_attempts: int = 3) -> str:
    attempts = 0
    while attempts < max_attempts:
        attempts += 1
        try:
            if attempts < 2:
                raise TimeoutError('Fake transient timeout')
            return f'success after {attempts} attempts'
        except TimeoutError:
            if attempts == max_attempts:
                raise
            await asyncio.sleep(0.01 * attempts)

print(await simulated_api_with_retry())

## 39. MCP (Model Context Protocol) tools

MAF can integrate MCP resources/tools. Use explicit allowlists, credentials and server trust checks; remote MCP endpoints can execute actions and return untrusted data. Exact MCP integration dependencies/version vary, so use current official runnable connector samples rather than assuming a generic constructor works everywhere.

Official repo: [MCP samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools) · [Agent Framework integrations](https://learn.microsoft.com/en-us/agent-framework/).

In [ ]:
MCP_POLICY = {
    'allowed_servers': ['approved-internal-mcp'],
    'allowed_tools': ['search_runbook', 'get_ticket'],
    'write_tools_require_human_approval': True,
}
assert 'delete_database' not in MCP_POLICY['allowed_tools']
print('Example MCP policy:', MCP_POLICY)

## 40. Agent-to-Agent (A2A) integration and distributed hosting

A2A is for interoperable agents across process/service boundaries. It is distinct from in-process orchestration and from MCP (tool protocol). Microsoft provides A2A hosting samples. Design authentication, authorization, replay protection, correlation IDs, transport-level timeouts and tracing end-to-end.

Official: [MAF hosting samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/04-hosting).

In [ ]:
A2A_MESSAGE = {
    'correlation_id': 'corr-' + uuid.uuid4().hex[:10],
    'source_agent': 'incident-triage',
    'target_agent': 'read-only-log-investigator',
    'task': 'Summarize redacted logs for INC-1001',
    'authz_scope': 'logs:read',
}
print(json.dumps(A2A_MESSAGE, indent=2))

## 41. Declarative YAML agent/workflow definitions

MAF includes **declarative workflow samples**: YAML can define process routes, conditions and agent actions. Keep YAML in source control, validate schema, restrict callable actions and avoid arbitrary dynamic expression execution.

Official samples: [Declarative workflows](https://github.com/microsoft/agent-framework/tree/main/python/samples/03-workflows/declarative).

In [ ]:
# Illustrative config design, NOT presented as an exact MAF declarative schema.
TEACHING_CONFIG = {'workflow': 'incident_analysis', 'steps': [
    {'id':'triage', 'mode':'deterministic'},
    {'id':'investigate', 'mode':'concurrent', 'agents':['logs','metrics','deployments']},
    {'id':'approve', 'mode':'human_gate'},
]}
print(json.dumps(TEACHING_CONFIG, indent=2))

## 42. Agent Harness and Agent Skills (Microsoft-specific)

The Microsoft documentation has **Agent Harness**, **Agent Skills** and **background agents** sections. These extend an agent runtime with task structure, contextual skills and execution environment concerns. They are not 1:1 LangGraph primitives; verify which capabilities are preview and their runtime/security constraints before teaching live.

Official: [MAF documentation landing](https://learn.microsoft.com/en-us/agent-framework/).

In [ ]:
SKILL_MANIFEST = {
    'skill': 'read_only_incident_summary',
    'description': 'Summarize supplied redacted incident evidence',
    'approved_actions': ['read', 'summarize'],
    'forbidden_actions': ['restart', 'delete', 'alter_iam'],
}
print('Illustrative skill policy (not vendor schema):', SKILL_MANIFEST)

## 43. RAG / enterprise search with MAF

An agent can call a retrieval tool which returns **approved, cited context**. Add ACL-aware document retrieval, tenant-level filtering, citations, retrieval evaluation and prompt-injection isolation. MAF integrations include vector stores and Azure services; a retrieval tool is not the same as the full RAG ingestion/indexing pipeline.

Source: [Vector store samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/vector_stores).

In [ ]:
RUNBOOKS = [
    {'id':'rb-001','service':'checkout','text':'For elevated 5xx inspect dependency timeouts and recent releases.'},
    {'id':'rb-002','service':'payments','text':'Inspect payment gateway timeout rate and rate limiting.'},
]

def retrieve_runbook(service: str) -> list[dict]:
    return [r for r in RUNBOOKS if r['service'] == service]

assert len(retrieve_runbook('checkout')) == 1
print(retrieve_runbook('checkout'))

## 44. Providers: Microsoft Foundry, Azure OpenAI, OpenAI, Ollama, others

Provider-agnostic design means your *agent orchestration* stays similar while client construction and model capabilities differ. Not every provider supports identical tool-call/structured-output/streaming features. MAF examples cover Microsoft Foundry, OpenAI and Ollama among others.

Official: [Provider sample directory](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/providers).

In [ ]:
PROVIDER_MATRIX = [
    {'backend':'OpenAI','client':'OpenAIChatClient','auth':'OPENAI_API_KEY'},
    {'backend':'Microsoft Foundry','client':'FoundryChatClient','auth':'AzureCliCredential / managed identity'},
    {'backend':'Ollama local','client':'OllamaChatClient','auth':'local service'},
]
for p in PROVIDER_MATRIX:
    print(f"{p['backend']}: {p['client']} ({p['auth']})")

## 45. Durable Task, Azure Functions, Microsoft Foundry hosting

For production execution, separate **agent runtime** from **hosting**. Possible patterns: long-lived API service, container jobs, message queues, Azure Functions / Durable Task integration or managed Foundry agent service. Workflow checkpointing != operational disaster recovery on its own.

Official: [MAF hosting samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/04-hosting) and [Microsoft docs](https://learn.microsoft.com/en-us/agent-framework/).

In [ ]:
PRODUCTION_ARCHITECTURE = {
    'entrypoint':'API / queue',
    'auth':'Entra ID or app identity + least privilege',
    'orchestration':'Microsoft Agent Framework workflow',
    'state':'approved checkpoint provider, encrypted and tenant scoped',
    'knowledge':'ACL-aware search/vector layer',
    'approvals':'human gate with auditable decision',
    'telemetry':'OpenTelemetry + redaction + alerting',
    'deploy':'containers / Azure Functions / Foundry as appropriate',
}
for k,v in PRODUCTION_ARCHITECTURE.items():
    print(f'{k:>15}: {v}')

## 46. Migrating AutoGen or Semantic Kernel to MAF

Microsoft Agent Framework is the successor direction for **AutoGen + Semantic Kernel**. Migration changes agent construction, conversation sessions, tools, routing and orchestration contracts; simply replacing imports is not sufficient.

Official: [AutoGen migration](https://learn.microsoft.com/en-us/agent-framework/migration-guide/from-autogen/) · [MAF repository](https://github.com/microsoft/agent-framework).

In [ ]:
MIGRATION_CHECKLIST = [
  'Inventory old AutoGen / Semantic Kernel components',
  'Replace old agent constructs with Agent(client=..., instructions=...)',
  'Map conversation state to AgentSession / context providers',
  'Rebuild multi-agent orchestration with MAF graph/builders',
  'Validate function signatures, @tool policies and structured outputs',
  'Test HITL pause/resume, retries, checkpoints, telemetry and security',
]
for index, item in enumerate(MIGRATION_CHECKLIST, 1):
    print(index, item)

## 47. Interview / instructor recap: state-of-practice concepts

**Teach these contrasts explicitly:**

1. Agent vs deterministic workflow vs multi-agent orchestration.
2. `AgentSession` conversation history vs **workflow checkpoint**.
3. `@tool`-based model-selected tool invocation vs Python-called deterministic tool functions.
4. MAF graph `Executor` + `WorkflowContext` vs LangGraph nodes/edges/reducers.
5. True handoff vs manager-led delegation vs group chat.
6. Checkpointing/replay vs at-least-once side effects, idempotency and distributed persistence.
7. Tool approval/HITL vs enterprise identity/authorization/guardrails.
8. Single-agent first vs the quality/cost tradeoff of many collaborating agents.

## 48. Assignments (classroom-ready)

- **Beginner:** Build a ticket-classifier MAF `Agent` with 2 mock read-only tools + Pydantic output.
- **Intermediate:** Migrate a LangGraph conditional-edge classifier into MAF `WorkflowBuilder` and test both branches.
- **Intermediate:** Write a functional workflow with parallel fan-out/fan-in and a bounded evaluator loop.
- **Advanced:** Add `AgentSession`, per-tenant encrypted memory, a secure retention/deletion policy, and session restore tests.
- **Advanced:** Add native MAF checkpoints and HITL pause/restart/replay; test tool-side-effect idempotency.
- **Capstone:** Build multi-agent incident resolution using sequential/concurrent orchestrators; include approval and verification gates.
- **Enterprise:** Add OpenTelemetry, RBAC, MCP read-only tool integration, eval cases, canary deployment plan, timeouts, budgets and disaster recovery.
- **Stretch:** Compare five MAF orchestrators for the same business task; report hallucination/error rate, latency, token usage, and auditability.

**Recommended teaching order:** 1–6 basics → 7–11 memory → 12–21 native workflows/HITL → 22–27 orchestration → 28–33 capstone → 34–46 production extras.

## 49. Source ledger — official Microsoft resources used

*All linked below are official Microsoft-owned repositories/documentation. Research accessed on 10 October 2026.*

1. [Agent Framework root repo](https://github.com/microsoft/agent-framework)
2. [Python install and quickstart](https://github.com/microsoft/agent-framework/blob/main/python/README.md)
3. [Python samples inventory](https://github.com/microsoft/agent-framework/blob/main/python/samples/AGENTS.md)
4. [Agent session and memory](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/)
5. [Structured output: Pydantic `response_format`](https://learn.microsoft.com/en-us/agent-framework/agents/structured-outputs)
6. [Workflow executor + edge starter](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/_start-here/step1_executors_and_edges.py)
7. [Conditional graph edge sample](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/control-flow/edge_condition.py)
8. [Functional HITL with `request_info`](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/functional/hitl_review.py)
9. [Functional `@step` and checkpoints](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/functional/steps_and_checkpointing.py)
10. [Checkpointing overview](https://learn.microsoft.com/en-us/agent-framework/workflows/checkpoints)
11. [Sub-workflow / WorkflowExecutor](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/composition/sub_workflow_basics.py)
12. [Official orchestration samples and builder patterns](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/orchestrations/README.md)
13. [Orchestration package README](https://github.com/microsoft/agent-framework/blob/main/python/packages/orchestrations/README.md)
14. [Tool approval and safety samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools)
15. [Context, middleware, providers, observability](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents)
16. [Workflows inventory: HITL, declarative, parallel, visualization](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/README.md)
17. [Self-hosting / A2A / Foundry hosting samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/04-hosting)
18. [Official AutoGen migration guide](https://learn.microsoft.com/en-us/agent-framework/migration-guide/from-autogen/)

### Accuracy notes

- This notebook was authored against **current official APIs** published at research cutoff. Vendor APIs evolve; match the installed SDK versions and retest live examples.
- Offline Python/Pydantic/SQLite labs can be tested without API credentials. **MAF imports, model calls, graph execution, and orchestration builder calls require installing the package; external model calls are not validated here unless explicitly stated.**
- The Azure/cloud vendor integration, MCP, A2A, Skills and hosting sections contain architecture demos and links rather than pretending that illustrative dictionaries are actual SDK configurations.
- Keep mock/simulated remediation disconnected from real infrastructure; never let an LLM authorize its own destructive action.